# Educational StyleGAN Face Lab (64 x 64)\n\nSelect **Runtime > Change runtime type > T4 GPU** before starting. This notebook trains the project's simplified StyleGAN-style network; the short smoke test uses synthetic images and is not CelebA training. Upload a ZIP of the complete project folder in the first cell, and an image ZIP in the dataset cell.\n

In [ ]:
from google.colab import files\nfrom pathlib import Path\nimport shutil\nuploaded = files.upload()  # Select a ZIP containing train.py, requirements.txt, models/, and utils/.\nproject_zips = [name for name in uploaded if name.lower().endswith('.zip')]\nif not project_zips:\n    raise FileNotFoundError('Upload a ZIP archive of the stylegan-celeba project.')\nPROJECT_ROOT = Path('/content/stylegan-celeba')\nPROJECT_ROOT.mkdir(parents=True, exist_ok=True)\nshutil.unpack_archive(project_zips[0], PROJECT_ROOT)\nif not (PROJECT_ROOT / 'train.py').is_file():\n    candidates = list(PROJECT_ROOT.rglob('train.py'))\n    if not candidates:\n        raise FileNotFoundError('Could not find train.py in the uploaded project ZIP.')\n    PROJECT_ROOT = candidates[0].parent\nprint('Project:', PROJECT_ROOT)

In [ ]:
%pip install -r "$PROJECT_ROOT/requirements.txt"

In [ ]:
from google.colab import drive\ndrive.mount('/content/drive')\nOUTPUT_DIR = '/content/drive/MyDrive/stylegan-celeba/outputs'\nimport torch\nprint('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())\nif not torch.cuda.is_available():\n    print('No GPU detected: smoke testing works, but face training will be much slower.')

In [ ]:
dataset_upload = files.upload()  # Select an image ZIP containing face JPG/PNG files.\nDATASET_ZIPS = [name for name in dataset_upload if name.lower().endswith('.zip')]\nif not DATASET_ZIPS:\n    raise FileNotFoundError('Upload a ZIP containing supported face image files.')\nDATASET_PATH = str(Path('/content') / DATASET_ZIPS[0])\nprint('Dataset ZIP:', DATASET_PATH)

In [ ]:
%cd $PROJECT_ROOT\nfrom dataset import load_dataset\nfrom utils.visualization import tensor_to_pil\nfrom IPython.display import display\ndataset = load_dataset(DATASET_PATH, resolution=64)\nprint('Preprocessed images:', len(dataset), '| tensor:', tuple(dataset[0].shape))\nfor index in range(min(8, len(dataset))):\n    display(tensor_to_pil(dataset[index]))

## Smoke test\nThis quick test checks a model update and saves outputs, using synthetic random images unless `DATASET_PATH` is supplied. It does not estimate CelebA quality.\n

In [ ]:
%cd $PROJECT_ROOT\n!python train.py --smoke-test --steps 1 --dataset "$DATASET_PATH" --output-dir "$OUTPUT_DIR/smoke"

## Full training\nThis cell can run for many hours. Start with a modest epoch count and batch size; reduce the batch size if GPU memory is insufficient. Keep the output directory on mounted Drive for persistence. A few epochs do not guarantee realistic faces.\n

In [ ]:
!python train.py --dataset "$DATASET_PATH" --epochs 100 --batch-size 8 --num-workers 2 --output-dir "$OUTPUT_DIR"

## Launch the interactive demo\nAfter training, run the following in a cell. `--share` creates a publicly accessible Gradio link; share it only if you intend to expose the demo.\n

In [ ]:
!python app.py --checkpoint "$OUTPUT_DIR/latest.pt" --share